# The Transformer's Three Attention Patterns: Self, Masked, and Cross

The full code of [The Transformer's Three Attention Patterns: Self, Masked, and Cross](https://residual-stream.lvtan71.workers.dev/posts/transformer-attention-patterns/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/transformer-attention-patterns) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/transformer-attention-patterns.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 1 min end to end on the RTX 3090 machine that saved the outputs below.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions change timings and the last digits of some floating-point results.
Sampled text and small training runs can move further; where that changes a
number the post discusses, the note above says so.

**Outputs saved from a run on NVIDIA GeForce RTX 3090 on 2026-10-18** (torch 2.14.1+cu130, 1 min; every cell ran). Run the notebook to replace them with your own.

In [1]:
%pip install -q numpy "torch==2.14.*" "transformers==5.19.*" huggingface_hub sentencepiece

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "transformer-attention-patterns"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

Note: you may need to restart the kernel to use updated packages.
working in residual-stream-code/posts/transformer-attention-patterns


## `attention.py`

The three patterns as one function, on the post's worked examples; then permutation equivariance.

In [2]:
!{sys.executable} attention.py

self-attention, 2 tokens
scores QK^T/sqrt(d):
[[0.000 0.707]
 [0.707 0.707]]
weights A:
[[0.330 0.670]
 [0.500 0.500]]
output AV:
[[0.330 1.000]
 [0.500 1.000]]

causal self-attention, 3 tokens
scores QK^T/sqrt(d):
[[ 0.000   -inf   -inf]
 [ 0.707  0.707   -inf]
 [-0.707  0.000 -0.707]]
weights A:
[[1.000 0.000 0.000]
 [0.500 0.500 0.000]
 [0.248 0.503 0.248]]
output AV:
[[1.000 1.000]
 [0.500 1.000]
 [0.497 0.752]]

cross-attention, 2 decoder tokens into 3 encoder tokens
scores QK^T/sqrt(d):
[[ 0.354  0.707 -0.354]
 [ 0.707  0.000  0.707]]
weights A:
[[0.343 0.488 0.169]
 [0.401 0.198 0.401]]
output AV:
[[0.512 0.831]
 [0.802 0.599]]



6 random tokens, rows shuffled by [5 4 2 1 3 0]: output rows shuffled the same way, largest difference 4.4e-16


The post printed:

```text
self-attention, 2 tokens
scores QK^T/sqrt(d):
[[0.000 0.707]
 [0.707 0.707]]
weights A:
[[0.330 0.670]
 [0.500 0.500]]
output AV:
[[0.330 1.000]
 [0.500 1.000]]

causal self-attention, 3 tokens
scores QK^T/sqrt(d):
[[ 0.000   -inf   -inf]
 [ 0.707  0.707   -inf]
 [-0.707  0.000 -0.707]]
weights A:
[[1.000 0.000 0.000]
 [0.500 0.500 0.000]
 [0.248 0.503 0.248]]
output AV:
[[1.000 1.000]
 [0.500 1.000]
 [0.497 0.752]]

cross-attention, 2 decoder tokens into 3 encoder tokens
scores QK^T/sqrt(d):
[[ 0.354  0.707 -0.354]
 [ 0.707  0.000  0.707]]
weights A:
[[0.343 0.488 0.169]
 [0.401 0.198 0.401]]
output AV:
[[0.512 0.831]
 [0.802 0.599]]

6 random tokens, rows shuffled by [5 4 2 1 3 0]: output rows shuffled the same way, largest difference 4.4e-16
```

## `scale.py`

Why attention divides by √d: the spread of q·k and what it does to a softmax.

In [3]:
!{sys.executable} scale.py

   d   std of q.k   std of q.k/sqrt(d)   softmax over 16 keys:  largest weight   gradient norm
                                                               raw    scaled     raw     scaled


   4         2.00                 1.00                   0.42     0.23    0.325    0.298


  16         4.02                 1.00                   0.68     0.24    0.279    0.303


  64         7.96                 0.99                   0.84     0.25    0.179    0.304


 256        15.98                 1.00                   0.93     0.25    0.096    0.305


1024        32.51                 1.02                   0.96     0.24    0.048    0.305


The post printed:

```text
   d   std of q.k   std of q.k/sqrt(d)   softmax over 16 keys:  largest weight   gradient norm
                                                               raw    scaled     raw     scaled
   4         2.00                 1.00                   0.42     0.23    0.325    0.298
  16         4.02                 1.00                   0.68     0.24    0.279    0.303
  64         7.96                 0.99                   0.84     0.25    0.179    0.304
 256        15.98                 1.00                   0.93     0.25    0.096    0.305
1024        32.51                 1.02                   0.96     0.24    0.048    0.305
```

## `causal.py`

Qwen2.5-1.5B-Instruct: the causal mask hides the future exactly; the strongest previous-token and first-token heads. Writes `heads.json`.

In [4]:
!{sys.executable} causal.py

17 tokens; changing the last one moves earlier positions' hidden states by at most 0.0e+00 (in every layer), and the last position's by 60.9


largest attention weight above the diagonal, any layer or head: 0.0e+00
average over all heads: weight on the previous token 0.13, on the first token 0.54
strongest previous token head: layer 1, head 4, average weight 0.78
strongest first token head: layer 14, head 5, average weight 1.00


The post printed:

```text
17 tokens; changing the last one moves earlier positions' hidden states by at most 0.0e+00 (in every layer), and the last position's by 60.9
largest attention weight above the diagonal, any layer or head: 0.0e+00
average over all heads: weight on the previous token 0.13, on the first token 0.54
strongest previous token head: layer 1, head 4, average weight 0.78
strongest first token head: layer 14, head 5, average weight 1.00
```

## `cross.py`

Marian English-to-German: cross-attention from each German token to the English ones, by layer. Writes `cross.json`.

In [5]:
!{sys.executable} cross.py

/root/venv/lib/python3.12/site-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


[transformers] Both `max_new_tokens` (=40) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


English: The black cat sleeps on the warm mat in the kitchen.
German:  Die schwarze Katze schläft auf der warmen Matte in der Küche.
14 German tokens attend over 14 English tokens (including </s>)
layer 0: each German token's most-weighted English token: ▁Die-></s> ▁schwarze->▁The ▁Katze-></s> ▁schläft-></s> ▁auf-></s> ▁der->▁on ▁warmen-></s> ▁Matt-></s> e-></s> ▁in-></s> ▁der->▁in ▁Küche-></s> .-></s> </s>->.
layer 1: each German token's most-weighted English token: ▁Die-></s> ▁schwarze->▁The ▁Katze->▁black ▁schläft->▁cat ▁auf->▁sleep ▁der->▁on ▁warmen->▁the ▁Matt-></s> e-></s> ▁in-></s> ▁der->▁in ▁Küche->▁the .-></s> </s>->.
layer 2: each German token's most-weighted English token: ▁Die-></s> ▁schwarze->▁The ▁Katze->▁black ▁schläft->▁cat ▁auf-></s> ▁der->▁on ▁warmen->▁the ▁Matt-></s> e-></s> ▁in->▁mat ▁der->▁in ▁Küche->▁kitchen .-></s> </s>->.
layer 3: each German token's most-weighted English token: ▁Die->. ▁schwarze->▁The ▁Katze->▁black ▁schläft->▁cat ▁auf->▁sleep ▁der->▁on ▁warmen

The post printed:

```text
English: The black cat sleeps on the warm mat in the kitchen.
German:  Die schwarze Katze schläft auf der warmen Matte in der Küche.
14 German tokens attend over 14 English tokens (including </s>)
layer 0: each German token's most-weighted English token: ▁Die-></s> ▁schwarze->▁The ▁Katze-></s> ▁schläft-></s> ▁auf-></s> ▁der->▁on ▁warmen-></s> ▁Matt-></s> e-></s> ▁in-></s> ▁der->▁in ▁Küche-></s> .-></s> </s>->.
layer 1: each German token's most-weighted English token: ▁Die-></s> ▁schwarze->▁The ▁Katze->▁black ▁schläft->▁cat ▁auf->▁sleep ▁der->▁on ▁warmen->▁the ▁Matt-></s> e-></s> ▁in-></s> ▁der->▁in ▁Küche->▁the .-></s> </s>->.
layer 2: each German token's most-weighted English token: ▁Die-></s> ▁schwarze->▁The ▁Katze->▁black ▁schläft->▁cat ▁auf-></s> ▁der->▁on ▁warmen->▁the ▁Matt-></s> e-></s> ▁in->▁mat ▁der->▁in ▁Küche->▁kitchen .-></s> </s>->.
layer 3: each German token's most-weighted English token: ▁Die->. ▁schwarze->▁The ▁Katze->▁black ▁schläft->▁cat ▁auf->▁sleep ▁der->▁on ▁warmen->▁the ▁Matt->▁warm e->▁mat ▁in->▁mat ▁der->▁in ▁Küche-></s> .-></s> </s>-></s>
layer 4: each German token's most-weighted English token: ▁Die->▁The ▁schwarze->▁black ▁Katze->▁cat ▁schläft->▁sleep ▁auf->▁on ▁der->▁the ▁warmen->▁warm ▁Matt->▁mat e-></s> ▁in->▁in ▁der-></s> ▁Küche->▁kitchen .-></s> </s>-></s>
layer 5: each German token's most-weighted English token: ▁Die-></s> ▁schwarze->▁black ▁Katze->▁cat ▁schläft->▁sleep ▁auf-></s> ▁der->▁warm ▁warmen->▁warm ▁Matt->▁mat e-></s> ▁in-></s> ▁der-></s> ▁Küche->▁kitchen .-></s> </s>-></s>
layer 4 is closest to a word-for-word diagonal
```

## `cost.py`

Causal attention at Qwen2.5-1.5B's shapes, timed from 512 to 8,192 tokens.

In [6]:
!{sys.executable} cost.py

tokens    time (ms)   x previous   score matrices (MiB)


   512         0.25                           6
  1024         0.52        2.1               24
  2048         1.72        3.3               96


  4096         7.01        4.1              384


  8192        26.09        3.7             1536


The post printed:

```text
tokens    time (ms)   x previous   score matrices (MiB)
   512         2.00                           6
  1024         6.13        3.1               24
  2048        17.64        2.9               96
  4096        70.37        4.0              384
  8192       289.81        4.1             1536
```